# Sesion 06 - Medallion Architecture: Capa Gold
## Laboratorio Practico | Databricks Data Engineer Associate

**Instructor:** Miguel Balcazar  
**Runtime minimo:** Databricks 13.3 LTS  
**Catalogo/Esquema:** `dbassociate.default`  
**Volume:** `/Volumes/dbassociate/default/vol_landing/`

---

### Objetivo del laboratorio

Construir la capa Gold de un pipeline Medallion para un dominio de ventas retail.  
El flujo parte de una tabla Silver certificada y produce una tabla Gold con:
- KPIs agregados por categoria, region y periodo
- Actualizacion incremental con MERGE INTO
- Optimizacion con OPTIMIZE ZORDER BY
- Validacion de data skipping con DESCRIBE DETAIL

**Duracion estimada:** 45-50 minutos


---
## PARTE 0 - Configuracion del entorno


In [0]:
import sys
print(f"Python version: {sys.version}")
spark.sql("SELECT current_catalog(), current_database()").show()

In [0]:
spark.sql("USE CATALOG dbassociate")
spark.sql("USE SCHEMA default")
print("Contexto establecido: dbassociate.default")

In [0]:
VOLUME_PATH = "/Volumes/dbassociate/default/vol_landing"
SILVER_PATH = f"{VOLUME_PATH}/sesion06/silver"
GOLD_PATH   = f"{VOLUME_PATH}/sesion06/gold"

dbutils.fs.mkdirs(SILVER_PATH)
dbutils.fs.mkdirs(GOLD_PATH)
print(f"Silver path: {SILVER_PATH}")
print(f"Gold path:   {GOLD_PATH}")

---
## PARTE 1 - Carga de datos Silver

Los archivos CSV del laboratorio representan transacciones de ventas retail ya procesadas
y estandarizadas (simulando una capa Silver certificada).

Debes subir los archivos `silver_ventas_batch1.csv` y `silver_ventas_batch2.csv`
al path: `/Volumes/dbassociate/default/vol_landing/sesion06/silver/`


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, TimestampType
)

schema_silver = StructType([
    StructField("venta_id",          StringType(),  False),
    StructField("fecha_venta",       DateType(),    False),
    StructField("anio",              IntegerType(), False),
    StructField("mes",               IntegerType(), False),
    StructField("trimestre",         IntegerType(), False),
    StructField("producto_id",       StringType(),  False),
    StructField("nombre_producto",   StringType(),  True),
    StructField("categoria",         StringType(),  True),
    StructField("subcategoria",      StringType(),  True),
    StructField("region",            StringType(),  True),
    StructField("tienda_id",         StringType(),  True),
    StructField("ciudad",            StringType(),  True),
    StructField("pais",              StringType(),  True),
    StructField("unidades",          IntegerType(), True),
    StructField("precio_unitario",   StringType(),  True),
    StructField("descuento_pct",     StringType(),  True),
    StructField("monto_bruto",       StringType(),  True),
    StructField("monto_neto",        StringType(),  True),
    StructField("costo_unitario",    StringType(),  True),
    StructField("margen_bruto",      StringType(),  True),
    StructField("cliente_id",        StringType(),  True),
    StructField("segmento_cliente",  StringType(),  True),
    StructField("canal_venta",       StringType(),  True),
    StructField("silver_created_at", TimestampType(),  True),
])

df_silver_raw = (
    spark.read
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(schema_silver)
    .csv(SILVER_PATH)
)

print(f"Registros leidos desde Silver: {df_silver_raw.count()}")
df_silver_raw.printSchema()

In [0]:
df_silver_raw.display()

In [0]:
from pyspark.sql.functions import col

# ANTIPATRON: usar DOUBLE para montos monetarios genera errores de punto flotante
# CORRECTO: cast explicito a DECIMAL con precision adecuada
df_silver = (
    df_silver_raw
    .withColumn("precio_unitario",   col("precio_unitario").cast("decimal(18,4)"))
    .withColumn("descuento_pct",     col("descuento_pct").cast("decimal(5,4)"))
    .withColumn("monto_bruto",       col("monto_bruto").cast("decimal(18,2)"))
    .withColumn("monto_neto",        col("monto_neto").cast("decimal(18,2)"))
    .withColumn("costo_unitario",    col("costo_unitario").cast("decimal(18,4)"))
    .withColumn("margen_bruto",      col("margen_bruto").cast("decimal(18,2)"))
    .withColumn("silver_created_at", col("silver_created_at").cast("timestamp"))
)

df_silver.printSchema()


In [0]:
df_silver.display()

In [0]:
df_silver.createOrReplaceTempView("vw_silver_ventas")
print("Vista temporal vw_silver_ventas registrada.")

In [0]:
df_silver.write.saveAsTable("dbassociate.default._vw_silver_ventas")

---
## PARTE 2 - Transformaciones Gold: KPIs por categoria, region y periodo

La tabla Gold `gold_kpi_ventas` agrega metricas de negocio con granularidad:
`anio` + `trimestre` + `mes` + `categoria` + `region`

Las metricas incluyen volumenes, montos, margenes y tasas de descuento.


In [0]:
df_gold_kpi = spark.sql(
    """
    SELECT
        anio, trimestre, mes,
        categoria, subcategoria,
        region, pais,
        canal_venta, segmento_cliente,

        COUNT(DISTINCT venta_id)                                        AS total_transacciones,
        SUM(unidades)                                                   AS total_unidades,
        COUNT(DISTINCT cliente_id)                                      AS clientes_unicos,
        COUNT(DISTINCT tienda_id)                                       AS tiendas_activas,

        CAST(SUM(monto_bruto)    AS DECIMAL(18,2))                      AS venta_bruta,
        CAST(SUM(monto_neto)     AS DECIMAL(18,2))                      AS venta_neta,
        CAST(SUM(margen_bruto)   AS DECIMAL(18,2))                      AS margen_bruto_total,
        CAST(AVG(precio_unitario) AS DECIMAL(18,4))                     AS precio_promedio,
        CAST(AVG(descuento_pct) * 100 AS DECIMAL(5,2))                  AS descuento_promedio_pct,
        CAST(
            SUM(margen_bruto) / NULLIF(SUM(monto_bruto), 0) * 100
            AS DECIMAL(5,2)
        )                                                               AS margen_pct,

        CURRENT_TIMESTAMP()                                             AS dw_updated_at
    FROM vw_silver_ventas
    GROUP BY
        anio, trimestre, mes,
        categoria, subcategoria,
        region, pais,
        canal_venta, segmento_cliente
    """
)

print(f"Registros Gold KPI: {df_gold_kpi.count()}")
df_gold_kpi.display()

---
## PARTE 3 - Window Functions: metricas de ventana analitica

Ademas de las agregaciones simples, Gold puede incluir metricas comparativas
calculadas con funciones de ventana (window functions).

Ejemplo: ranking de categorias por venta dentro de cada region y periodo,
YTD acumulado, y variacion respecto al mes anterior.


In [0]:
from pyspark.sql.functions import rank, dense_rank, lag, sum as spark_sum, round as spark_round, when
from pyspark.sql.window import Window

# Window: ranking dentro de region y mes por venta neta descendente
w_region_mes = (
    Window
    .partitionBy("region", "anio", "mes")
    .orderBy(col("venta_neta").desc())
)


# Window: YTD acumulado por region y anio
w_ytd = (
    Window
    .partitionBy("region", "anio")
    .orderBy("mes")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Window: comparacion mes anterior por region y categoria
w_lag = (
    Window
    .partitionBy("region", "categoria", "anio")
    .orderBy("mes")
)


df_gold_analitico = (
    df_gold_kpi
    .withColumn("rank_categoria_region_mes", rank().over(w_region_mes))
    .withColumn("dense_rank_categoria",      dense_rank().over(w_region_mes))
    .withColumn("venta_neta_ytd",
        spark_sum("venta_neta").over(w_ytd).cast("decimal(18,2)"))
    .withColumn("venta_neta_mes_anterior",
        lag("venta_neta", 1).over(w_lag).cast("decimal(18,2)"))
    .withColumn("variacion_mes_pct",
        when(
            col("venta_neta_mes_anterior").isNotNull() & (col("venta_neta_mes_anterior") != 0),
            spark_round(
                (col("venta_neta") - col("venta_neta_mes_anterior"))
                / col("venta_neta_mes_anterior") * 100,
                2
            )
        ).otherwise(None)
    )
)

# Top 3 categorias por region en el primer mes del dataset
(df_gold_analitico
    .filter((col("rank_categoria_region_mes") <= 3) & (col("mes") == 1))
    .select(
        "anio", "mes", "region", "categoria",
        "venta_neta", "rank_categoria_region_mes",
        "venta_neta_ytd", "variacion_mes_pct"
    )
    .orderBy("region", "rank_categoria_region_mes")
    .display()
)

In [0]:
df_gold_analitico.display()

---
## PARTE 4 - Escritura inicial en Gold (primera carga)

La primera escritura usa overwrite para crear la tabla Gold base.
Las cargas siguientes usaran MERGE INTO para actualizacion incremental.

**Particionamiento:** por `anio` y `region` solamente.
ANTIPATRON: particionar por fecha completa (anio+mes+dia) genera cientos de particiones
pequeñas que degradan el rendimiento de listing en tablas de tamanio moderado.


In [0]:
(
    df_gold_analitico.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("anio", "region")
    .saveAsTable("dbassociate.default.gold_kpi_ventas")
)

print("Tabla gold_kpi_ventas creada.")
spark.sql("DESCRIBE EXTENDED dbassociate.default.gold_kpi_ventas").display()

In [0]:
# Documentar la tabla y columnas clave
spark.sql(
    "COMMENT ON TABLE dbassociate.default.gold_kpi_ventas IS "
    "'Tabla Gold de KPIs de ventas retail. Granularidad: anio/trimestre/mes/categoria/region. "
    "Actualizada por MERGE incremental desde Silver certificado.'"
)

spark.sql(
    "ALTER TABLE dbassociate.default.gold_kpi_ventas "
    "ALTER COLUMN venta_neta COMMENT 'Monto neto de ventas en USD despues de descuentos. DECIMAL(18,2).'"
)


spark.sql(
    "ALTER TABLE dbassociate.default.gold_kpi_ventas "
    "ALTER COLUMN margen_pct COMMENT 'Margen bruto como porcentaje de venta bruta. Calculado: margen_bruto_total/venta_bruta*100.'"
)
spark.sql(
    "ALTER TABLE dbassociate.default.gold_kpi_ventas "
    "ALTER COLUMN dw_updated_at COMMENT 'Timestamp de la ultima actualizacion de este registro en la capa Gold.'"
)

print("Documentacion registrada.")

---
## PARTE 5 - Actualizacion incremental con MERGE INTO

En produccion, Gold se actualiza con nuevos datos desde Silver sin reescribir toda la tabla.
MERGE INTO permite actualizar registros existentes e insertar nuevos en una sola operacion.

La clave de negocio de esta tabla Gold es la combinacion de todas las dimensiones de agregacion.


In [0]:
count_antes = spark.sql("SELECT COUNT(*) AS total FROM dbassociate.default.gold_kpi_ventas").collect()[0]["total"]
print(f"Registros en Gold ANTES del merge: {count_antes}")

In [0]:
%sql
SELECT * FROM dbassociate.default.gold_kpi_ventas WHERE mes = 1 LIMIT 10

In [0]:
# Simular recalculo de un mes con datos actualizados (en produccion vendria de Silver filtrado por batch)
from pyspark.sql.functions import current_timestamp, lit

df_recalculo = (
    spark.sql(
        "SELECT * FROM dbassociate.default.gold_kpi_ventas WHERE mes = 1 LIMIT 10"
    )
    .withColumn("venta_neta",        (col("venta_neta") * lit("1.05")).cast("decimal(18,2)"))
    .withColumn("margen_bruto_total",(col("margen_bruto_total") * lit("1.05")).cast("decimal(18,2)"))
    .withColumn("dw_updated_at",     current_timestamp())
)


df_recalculo.createOrReplaceTempView("vw_gold_recalculo")
print(f"Registros a mergear: {df_recalculo.count()}")

In [0]:
spark.sql(
    """
    MERGE INTO dbassociate.default.gold_kpi_ventas AS target
    USING vw_gold_recalculo AS source
    ON (
        target.anio              = source.anio
        AND target.trimestre     = source.trimestre
        AND target.mes           = source.mes
        AND target.categoria     = source.categoria
        AND target.subcategoria  = source.subcategoria
        AND target.region        = source.region
        AND target.canal_venta   = source.canal_venta
        AND target.segmento_cliente = source.segmento_cliente
    )
    WHEN MATCHED THEN UPDATE SET
        target.total_transacciones    = source.total_transacciones,
        target.total_unidades         = source.total_unidades,
        target.clientes_unicos        = source.clientes_unicos,
        target.venta_bruta            = source.venta_bruta,
        target.venta_neta             = source.venta_neta,
        target.margen_bruto_total     = source.margen_bruto_total,
        target.precio_promedio        = source.precio_promedio,
        target.descuento_promedio_pct = source.descuento_promedio_pct,
        target.margen_pct             = source.margen_pct,
        target.venta_neta_ytd         = source.venta_neta_ytd,
        target.variacion_mes_pct      = source.variacion_mes_pct,
        target.dw_updated_at          = source.dw_updated_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

count_despues = spark.sql("SELECT COUNT(*) AS total FROM dbassociate.default.gold_kpi_ventas").collect()[0]["total"]
print(f"Registros despues del merge: {count_despues}")
print(f"Nuevos inserts netos:        {count_despues - count_antes}")

---
## PARTE 6 - OPTIMIZE y ZORDER BY

Despues de cargas incrementales, la tabla Gold acumula small files.
OPTIMIZE los compacta; ZORDER BY mejora el data skipping en consultas filtradas.

Elegimos `categoria` y `mes` para Z-Order porque:
- Las consultas Gold suelen filtrar por categoria (analisis de producto)
- mes es columna frecuente en filtros de periodo
- `region` y `anio` ya estan en el particionamiento fisico


In [0]:
detail_antes = spark.sql("DESCRIBE DETAIL dbassociate.default.gold_kpi_ventas").collect()[0]
print(f"Archivos antes de OPTIMIZE: {detail_antes['numFiles']}")
print(f"Bytes totales: {detail_antes['sizeInBytes']:,}")
print(f"Columnas de particion: {detail_antes['partitionColumns']}")

In [0]:
spark.sql(
    "OPTIMIZE dbassociate.default.gold_kpi_ventas ZORDER BY (categoria, mes)"
)

detail_despues = spark.sql("DESCRIBE DETAIL dbassociate.default.gold_kpi_ventas").collect()[0]
print(f"Archivos despues de OPTIMIZE: {detail_despues['numFiles']}")
print(f"Bytes totales: {detail_despues['sizeInBytes']:,}")

In [0]:
%sql

select * from dbassociate.default._vw_silver_ventas

In [0]:
%sql
OPTIMIZE dbassociate.default._vw_silver_ventas ZORDER BY (venta_id, producto_id)

In [0]:
%sql
OPTIMIZE dbassociate.default.gold_kpi_ventas

---
## PARTE 7 - Validacion de data skipping

Ejecutar una consulta con filtros selectivos para activar el data skipping de Delta.
Las estadisticas del transaction log permiten omitir archivos que no pueden contener
los valores del filtro.


In [0]:
spark.sql(
    """
    SELECT
        categoria, region,
        SUM(venta_neta)     AS total_venta_neta,
        SUM(total_unidades) AS total_unidades,
        AVG(margen_pct)     AS margen_promedio
    FROM dbassociate.default.gold_kpi_ventas
    WHERE anio = 2024
      AND mes BETWEEN 1 AND 3
      AND categoria = 'Electronica'
    GROUP BY categoria, region
    ORDER BY total_venta_neta DESC
    """
).display()

In [0]:
# Verificar operaciones registradas en el history de la tabla
spark.sql(
    """
    DESCRIBE HISTORY dbassociate.default.gold_kpi_ventas
    """
).select(
    "version", "timestamp", "operation", "operationMetrics"
).display()

---
## PARTE 8 - Vista de publicacion para BI

Crear una vista sobre Gold con nombres de columna descriptivos
para exponer a herramientas de BI sin revelar columnas tecnicas internas.

ANTIPATRON: exponer columnas de auditoria interna (dw_updated_at, rank interno, etc.)
directamente a consumidores de BI.


In [0]:
spark.sql(
    """
    CREATE OR REPLACE VIEW dbassociate.default.vw_gold_ventas_bi AS
    SELECT
        anio                                            AS Anio,
        trimestre                                       AS Trimestre,
        mes                                             AS Mes,
        categoria                                       AS Categoria,
        subcategoria                                    AS Subcategoria,
        region                                          AS Region,
        pais                                            AS Pais,
        canal_venta                                     AS Canal_Venta,
        segmento_cliente                                AS Segmento_Cliente,
        total_transacciones                             AS Total_Transacciones,
        total_unidades                                  AS Total_Unidades,
        clientes_unicos                                 AS Clientes_Unicos,
        tiendas_activas                                 AS Tiendas_Activas,
        CAST(venta_bruta        AS DECIMAL(18,2))       AS Venta_Bruta_USD,
        CAST(venta_neta         AS DECIMAL(18,2))       AS Venta_Neta_USD,
        CAST(margen_bruto_total AS DECIMAL(18,2))       AS Margen_Bruto_USD,
        CAST(margen_pct         AS DECIMAL(5,2))        AS Margen_Pct,
        CAST(descuento_promedio_pct AS DECIMAL(5,2))    AS Descuento_Promedio_Pct,
        CAST(venta_neta_ytd     AS DECIMAL(18,2))       AS Venta_Neta_YTD_USD,
        CAST(variacion_mes_pct  AS DECIMAL(5,2))        AS Variacion_Mes_Pct
    FROM dbassociate.default.gold_kpi_ventas
    """
)

spark.sql(
    "COMMENT ON TABLE dbassociate.default.vw_gold_ventas_bi IS "
    "'Vista de publicacion para BI. KPIs de ventas retail con nombres descriptivos. No usar en pipelines internos.'"
)

print("Vista vw_gold_ventas_bi creada.")
spark.sql("SELECT COUNT(*) AS registros FROM dbassociate.default.vw_gold_ventas_bi").show()

---
## PARTE 9 - Gobierno: permisos en Unity Catalog

Bloque de referencia para aplicar permisos en entornos con Unity Catalog y privilegios de administrador.

En produccion:
- Otorgar permisos a grupos, nunca a usuarios individuales
- GRANT USE CATALOG y USE SCHEMA son requisitos previos a GRANT SELECT sobre tablas


In [0]:
# BLOQUE DE GOBIERNO - requiere privilegios de administrador en Unity Catalog
# Descomentar en entornos con permisos disponibles

# spark.sql("GRANT USE CATALOG ON CATALOG dbassociate TO `analysts`")
# spark.sql("GRANT USE SCHEMA ON SCHEMA dbassociate.default TO `analysts`")
# spark.sql("GRANT SELECT ON TABLE dbassociate.default.gold_kpi_ventas TO `analysts`")
# spark.sql("GRANT SELECT ON TABLE dbassociate.default.vw_gold_ventas_bi TO `analysts`")
# spark.sql("SHOW GRANTS ON TABLE dbassociate.default.gold_kpi_ventas").show(truncate=False)

print("Bloque de gobierno comentado.")
print("En produccion: usar grupos en GRANT, nunca usuarios individuales.")

---
## PARTE 10 - Validacion final y resumen


In [0]:
print("=" * 60)
print("RESUMEN DEL LABORATORIO - CAPA GOLD")
print("=" * 60)

spark.sql("SHOW TABLES IN dbassociate.default LIKE 'gold*|vw_gold*'").show(truncate=False)

total = spark.sql(
    "SELECT COUNT(*) AS total FROM dbassociate.default.gold_kpi_ventas"
).collect()[0]["total"]
print(f"Registros en gold_kpi_ventas: {total}")

In [0]:
spark.sql(
    """
    SELECT
        anio, mes, categoria, region,
        total_transacciones, venta_neta,
        margen_pct, variacion_mes_pct,
        rank_categoria_region_mes
    FROM dbassociate.default.gold_kpi_ventas
    WHERE rank_categoria_region_mes <= 2
    ORDER BY anio, mes, region, rank_categoria_region_mes
    LIMIT 20
    """
).show(truncate=False)

---
## LIMPIEZA (opcional)

Ejecutar solo al finalizar si se desea limpiar el entorno de laboratorio.


In [0]:
# LIMPIEZA - descomenta para ejecutar
# spark.sql("DROP TABLE IF EXISTS dbassociate.default.gold_kpi_ventas")
# spark.sql("DROP VIEW IF EXISTS dbassociate.default.vw_gold_ventas_bi")
# dbutils.fs.rm(f"{VOLUME_PATH}/sesion06", recurse=True)
# print("Entorno limpiado.")
print("Limpieza comentada.")